# Metal3D Prediction

![Metal3D Prediction](https://proto-bio.github.io/proto-assets/images/tool/metal3d/hero.png)

This notebook demonstrates `run_metal3d_prediction`, which predicts zinc-ion binding sites in a protein structure. It scores candidate metal-coordinating residues, returns clustered metal-site coordinates with a per-site confidence, and appends the top predicted zinc to an annotated structure.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("metal3d")
display_overview("metal3d")
display_docs_section("metal3d", "Background")

# Metal3D

Metal3D is a deep-learning model that predicts where zinc ions bind in a protein structure, developed by the Laboratory of Computational Chemistry and Biochemistry at EPFL. Given a structure, it scores candidate metal-binding sites and returns the predicted zinc coordinates together with a per-site confidence. This toolkit runs Metal3D over one or more input structures, returning clustered metal-site probabilities, optional per-residue probabilities, and an annotated PDB containing the top predicted zinc site when it clears the reporting threshold.

Metal3D ([Dürr, Levy, and Rothlisberger, 2023](https://doi.org/10.1038/s41467-023-37870-6)) is a three-dimensional convolutional neural network for predicting zinc-ion locations in protein structures. Around each candidate metal-coordinating residue (such as histidine, cysteine, aspartate, and glutamate), it voxelizes the local atomic environment into a grid of physicochemical features — capturing properties such as hydrophobicity, aromaticity, metal-coordinating atoms, hydrogen-bond donors and acceptors, and charge. The network maps each voxelized environment to a per-voxel probability of zinc occupancy; these residue-centered densities are averaged onto a shared grid and clustered into discrete predicted sites, each with a confidence value. Because it reasons from local structure rather than sequence conservation, Metal3D localizes zinc ions accurately even for proteins with few homologs in the Protein Data Bank.

Metal3D yields two complementary outputs used in protein engineering: a per-residue zinc density that feeds into design workflows, and a global zinc density suitable for annotating computationally predicted structures. The published model is trained solely on zinc sites from the Protein Data Bank, though the authors note that the same framework extends to other metals by retraining on the corresponding sites.

This toolkit defaults to the published checkpoint (`metal3d-original`) and additionally bundles two retrained variants from dEVA ([El Nesr et al., 2026](https://www.biorxiv.org/content/10.1101/2026.04.23.720277)), a multi-objective protein-design framework that uses Metal3D to score catalytic-metal coordination: `metal3d-cat` and `metal3d-clean`. These variants adopt a slightly modified network architecture and a wider grid-averaging radius; all three checkpoints are downloaded from the [dEVA repository](https://github.com/gelnesr/dEVA) during standalone setup.

## Available tools

In [2]:
display_available_tools("metal3d")

- **`run_metal3d_prediction()`** — Predict catalytic or structural metal-ion sites in protein structures using Metal3D/dEVA checkpoints.

### `run_metal3d_prediction`

Metal3D scores candidate metal-coordinating residues and predicts where zinc ions bind. Pass `candidate_residues` to restrict scoring to a known pocket and obtain per-residue probabilities, or omit it to evaluate all metal-binding residue types across the protein. The default `metal3d-original` checkpoint is the published Metal3D model; `metal3d-cat` and `metal3d-clean` are dEVA's retrained variants.

In [3]:
from pathlib import Path

from proto_tools.tools.structure_scoring.metal3d import (
    Metal3DPredictionConfig,
    Metal3DPredictionInput,
    Metal3DStructureInput,
    run_metal3d_prediction,
)
from proto_tools.entities.structures import Structure

In [4]:
# Display input docs
display_api_reference("metal3d", "input", "run_metal3d_prediction")

# Carbonic anhydrase (2VVB) — score the His94/His96/His119 zinc-coordinating triad
structure = Structure(structure=str(Path("..") / "example_input_fixture.pdb"))

inputs = Metal3DPredictionInput(
    inputs=[Metal3DStructureInput(structure=structure, candidate_residues={"X": [94, 96, 119]})]
)

**Input** — `Metal3DPredictionInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>inputs</code> | <code>list[Metal3DStructureInput]</code> | required | Structures to evaluate with Metal3D. |

In [5]:
# Display config docs
display_api_reference("metal3d", "config", "run_metal3d_prediction")

# Configure prediction | see docs above for all fields
config = Metal3DPredictionConfig(
    model_checkpoint="metal3d-original",
    probability_threshold=0.2,
    device="cuda",
)

**Config** — `Metal3DPredictionConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cuda'</code> | Device to run the model on. |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>model_checkpoint</code> | <code>Literal['metal3d-original', 'metal3d-cat', 'metal3d-clean']</code> | <code>'metal3d-original'</code> | Metal3D checkpoint: original Metal3D, or dEVA's metal3d-cat/metal3d-clean variants. |
| <code>probability_threshold</code> | <code>float</code> | <code>0.2</code> | Threshold for reporting and annotating predicted metal sites. |
| <code>cluster_distance_threshold</code> | <code>float</code> | <code>7.0</code> | Maximum complete-linkage distance in Angstroms for merging high-probability grid points. |
| <code>max_sites</code> | <code>int</code> | <code>8</code> | Maximum number of clustered sites to return per structure. |

In [6]:
# Run the prediction function
result = run_metal3d_prediction(inputs, config)

Running run_metal3d_prediction [00:00]

In [7]:
# Display output docs
display_api_reference("metal3d", "output", "run_metal3d_prediction")

# Print the top predicted zinc site and per-residue probabilities
prediction = result.results[0]
print(f"Site found: {prediction.found} | p(metal): {prediction['pmetal']:.3f}")
for site in prediction.sites:
    print(f"  Zn site: ({site.x:.2f}, {site.y:.2f}, {site.z:.2f})  p={site['probability']:.3f}")
for rp in prediction.residue_probabilities:
    print(f"  {rp.residue_name}{rp.residue_id} (chain {rp.chain_id}): p={rp['probability']:.3f}")

**Output** — `Metal3DPredictionOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>results</code> | <code>list[Metal3DPredictionResult]</code> | <code>[]</code> | One Metal3D prediction result per input structure. |

**Metrics** — `Metal3DPredictionResult` (one set per `results` item)

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>pmetal</code> **(primary)** | <code>float</code> | <code>[0, 1]</code> |  |  |

Site found: True | p(metal): 0.997
  Zn site: (-6.91, -1.35, 15.57)  p=0.997
  HIS94 (chain X): p=0.997
  HIS96 (chain X): p=0.998
  HIS119 (chain X): p=0.998


## Export Results

Predictions can be exported to JSON, or the annotated structures (with the predicted zinc appended) to PDB.

In [8]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export(name="metal3d_sites", export_path=output_dir, file_format="json")
print(f"Predictions exported to {output_dir / 'metal3d_sites.json'}")

result.export(name="metal3d_annotated", export_path=output_dir, file_format="pdb")
print(f"Annotated structures exported to {output_dir / 'metal3d_annotated/'}")

Predictions exported to example_output/metal3d_sites.json
Annotated structures exported to example_output/metal3d_annotated
